# ICDAR Writer Classification

Fine-tune the pen-classification backbones (`resnet18`, `convnext_tiny`) on the ICDAR 2026 writer-identification competition using the same augmentations and training loop as the pen notebook, but with a lower learning rate and writer-specific prediction mapping.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, convnext_tiny


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# Data Loading

In [ ]:
DATASET_DIR = Path("/kaggle/input/competitions/icdar-2026-circleid-writer-identification")
PEN_MODEL_DIR = Path("/kaggle/input/icdar-pen-backbones")
OUTPUT_DIR = Path("/kaggle/working/icdar_writer_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATASET_DIR / "train.csv")
additional_path = DATASET_DIR / "additional_train.csv"
if additional_path.exists():
    additional_df = pd.read_csv(additional_path)
    train_df = pd.concat([train_df, additional_df], ignore_index=True)

test_df = pd.read_csv(DATASET_DIR / "test.csv")
sample_submission = pd.read_csv(DATASET_DIR / "sample_submission.csv")

print(f"train_df shape: {train_df.shape}")
print(f"test_df shape: {test_df.shape}")
train_df.head()


In [ ]:
def ensure_image_path(df):
    df = df.copy()

    if "image_path" in df.columns:
        df["image_path"] = df["image_path"].astype(str)
        if len(df) and not df["image_path"].iloc[0].startswith("/"):
            df["image_path"] = df["image_path"].map(lambda x: str(DATASET_DIR / x))
    else:
        df["image_path"] = df["image_id"].astype(str).map(
            lambda x: str(DATASET_DIR / "images" / f"{x}.png")
        )

    return df

train_df = ensure_image_path(train_df)
test_df = ensure_image_path(test_df)


In [ ]:
writer_ids = sorted(train_df["writer_id"].unique().tolist())
writer_to_label = {writer_id: idx for idx, writer_id in enumerate(writer_ids)}
label_to_writer = {idx: writer_id for writer_id, idx in writer_to_label.items()}
NUM_CLASSES = len(writer_ids)

train_df["label"] = train_df["writer_id"].map(writer_to_label)

print("Number of writers:", NUM_CLASSES)
print(train_df[["writer_id", "label"]].head())


# Train/validation split

In [ ]:
train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["label"]
)

train_split_df = train_split_df.reset_index(drop=True)
val_split_df = val_split_df.reset_index(drop=True)

print(train_split_df.shape, val_split_df.shape)


# Dataset creation

In [ ]:
class CircleDataset(Dataset):

    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, row["image_id"]

        label = row["label"]
        return image, label


# Image Transformation

In [ ]:
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.95, 1.0),
        ratio=(0.95, 1.05)
    ),
    transforms.RandomRotation(12),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.97, 1.03)
    ),
    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.08
    ),
    transforms.ToTensor(),
    normalize
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])


# Loading fine-tuning models

In [ ]:
PRETRAINED_PATHS = {
    "resnet": PEN_MODEL_DIR / "resnet_pen_best.pth",
    "convnext": PEN_MODEL_DIR / "convnext_pen_best.pth",
}


def get_checkpoint_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["state_dict", "model_state_dict", "model"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                checkpoint = checkpoint[key]
                break
    cleaned = {}
    for key, value in checkpoint.items():
        key = key.replace("module.", "")
        cleaned[key] = value
    return cleaned


def load_backbone_weights(model, checkpoint_path, model_name):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        print(f"⚠️ Missing pretrained checkpoint for {model_name}: {checkpoint_path}")
        return model

    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    state_dict = get_checkpoint_state_dict(checkpoint)

    if model_name == "resnet":
        filtered_state_dict = {
            key: value
            for key, value in state_dict.items()
            if not key.startswith("fc.")
        }
    else:
        filtered_state_dict = {
            key: value
            for key, value in state_dict.items()
            if not key.startswith("classifier.2")
        }

    missing, unexpected = model.load_state_dict(filtered_state_dict, strict=False)
    print(f"Loaded {model_name} backbone from {checkpoint_path.name}")
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)
    return model


def build_model(model_name="resnet", n_classes=NUM_CLASSES):

    if model_name == "resnet":
        model = resnet18(weights="IMAGENET1K_V1")
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, n_classes)
        )

    elif model_name == "convnext":
        model = convnext_tiny(weights="IMAGENET1K_V1")
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, n_classes)
        )

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    model = load_backbone_weights(model, PRETRAINED_PATHS[model_name], model_name)
    return model


# Model Training

In [ ]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = F.cross_entropy(logits, labels, label_smoothing=0.05)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, device):
    model.eval()

    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        preds = logits.argmax(1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return correct / total


In [ ]:
BATCH_SIZE = 96
EPOCHS_resnet = 6
EPOCHS_convnext = 4
LR = 1e-4
NUM_WORKERS = 2

models = []
model_paths = {}

train_ds = CircleDataset(train_split_df, train_tfms)
val_ds = CircleDataset(val_split_df, val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

for model_name in ["resnet", "convnext"]:

    print(f"Training {model_name}")

    model = build_model(model_name, NUM_CLASSES).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=1e-4
    )

    if model_name == "resnet":
        EPOCHS = EPOCHS_resnet
    else:
        EPOCHS = EPOCHS_convnext

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    best_val_acc = 0.0
    best_path = OUTPUT_DIR / f"writer_{model_name}_best.pth"

    for epoch in range(EPOCHS):

        train_loss = train_epoch(model, train_loader, optimizer, device)
        val_acc = validate(model, val_loader, device)

        print(f"{model_name} Epoch {epoch} loss {train_loss:.4f} val_acc {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)

        scheduler.step()

    print(f"Best {model_name} val_acc: {best_val_acc:.4f}")

    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()

    model_paths[model_name] = str(best_path)
    models.append(model)


# Inference

In [ ]:
test_ds = CircleDataset(test_df, val_tfms, is_test=True)

test_loader = DataLoader(
    test_ds,
    batch_size=128,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

preds = []

for model in models:

    model.eval()
    fold_preds = []

    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)

            probs = []

            logits = model(images)
            probs.append(torch.softmax(logits, 1))

            logits = model(torch.flip(images, dims=[3]))
            probs.append(torch.softmax(logits, 1))

            logits = model(torch.flip(images, dims=[2]))
            probs.append(torch.softmax(logits, 1))

            logits = model(torch.rot90(images, 1, [2, 3]))
            probs.append(torch.softmax(logits, 1))

            prob = torch.mean(torch.stack(probs), dim=0)
            fold_preds.append(prob.cpu().numpy())

    preds.append(np.concatenate(fold_preds))

preds = np.mean(preds, axis=0)
label_preds = preds.argmax(1)
writer_preds = [label_to_writer[idx] for idx in label_preds]


# Submission

In [ ]:
submission = sample_submission.copy()
target_column = "writer_id" if "writer_id" in submission.columns else submission.columns[-1]
submission[target_column] = writer_preds
submission.to_csv("submission.csv", index=False)

submission.head()
